In [1]:
import numpy as np
import pandas as pd
from math import pi
import warnings
warnings.filterwarnings('ignore')

import statsmodels.api as sm
import statsmodels.formula.api as smf
import sklearn.linear_model as sklm
import matplotlib.pyplot as plt

In [2]:
# 选取的股票为600600
# 读取并拼接日度数据
data1 = pd.read_excel('./data2/日度数据/RESSET_DRESSTK_2001_2010_1.xls', usecols=[0,1,2,3])
data2 = pd.read_excel('./data2/日度数据/RESSET_DRESSTK_2011_2015_1.xls', usecols=[0,1,2,3])
data3 = pd.read_excel('./data2/日度数据/RESSET_DRESSTK_2016_2020_1.xls', usecols=[0,1,2,3])
data4 = pd.read_excel('./data2/日度数据/RESSET_DRESSTK_2021__1.xls', usecols=[0,1,2,3])
data_daily = pd.concat([data1,data2],ignore_index=True)
data_daily = pd.concat([data_daily,data3],ignore_index=True)
data_daily = pd.concat([data_daily,data4],ignore_index=True)
data_daily

,股票代码_Stkcd,日期_Date,收盘价_Clpr,日收益率_Dret
0,600600,2005-05-30,8.88,0.0278
1,600600,2005-05-31,8.74,-0.0158
2,600600,2005-06-01,8.65,-0.0103
3,600600,2005-06-02,8.69,0.0046
4,600600,2005-06-03,8.58,-0.0127
...,...,...,...,...
5358,600600,2022-12-26,106.38,-0.0295
5359,600600,2022-12-27,108.83,0.0230
5360,600600,2022-12-28,108.76,-0.0006
5361,600600,2022-12-29,107.15,-0.0148


In [3]:
# 改列名
data_daily.columns = ['stock','date','close','Dret']

# 转换一下日期表示
data_daily['date'] = pd.to_datetime(data_daily['date'])
data_daily['yearmonth'] = data_daily['date'].dt.strftime('%Y%m').astype(int)

data_daily

,stock,date,close,Dret,yearmonth
0,600600,2005-05-30,8.88,0.0278,200505
1,600600,2005-05-31,8.74,-0.0158,200505
2,600600,2005-06-01,8.65,-0.0103,200506
3,600600,2005-06-02,8.69,0.0046,200506
4,600600,2005-06-03,8.58,-0.0127,200506
...,...,...,...,...,...
5358,600600,2022-12-26,106.38,-0.0295,202212
5359,600600,2022-12-27,108.83,0.0230,202212
5360,600600,2022-12-28,108.76,-0.0006,202212
5361,600600,2022-12-29,107.15,-0.0148,202212


In [4]:
# 去除日度数据中有nan值的行
data_daily.dropna(inplace = True)
data_daily

,stock,date,close,Dret,yearmonth
0,600600,2005-05-30,8.88,0.0278,200505
1,600600,2005-05-31,8.74,-0.0158,200505
2,600600,2005-06-01,8.65,-0.0103,200506
3,600600,2005-06-02,8.69,0.0046,200506
4,600600,2005-06-03,8.58,-0.0127,200506
...,...,...,...,...,...
5358,600600,2022-12-26,106.38,-0.0295,202212
5359,600600,2022-12-27,108.83,0.0230,202212
5360,600600,2022-12-28,108.76,-0.0006,202212
5361,600600,2022-12-29,107.15,-0.0148,202212


In [5]:
# 读取月度数据
data_monthly = pd.read_excel('./data2/monthly_data.xls',usecols=[0,1,2,3,4,5,6,7,8,9])

data_monthly.columns = ['stock','date','Trdsum','MonTurnR','Monret','Monrfret','PE','EPS','ROE','IncomePS']

data_monthly['date'] = pd.to_datetime(data_monthly['date'])
data_monthly['yearmonth'] = data_monthly['date'].dt.strftime('%Y%m').astype(int)

data_monthly.dropna(inplace = True)
data_monthly

,stock,date,Trdsum,MonTurnR,Monret,Monrfret,PE,EPS,ROE,IncomePS,yearmonth
0,600600,2001-01-19,1.919654e+08,3.1578,0.0112,0.001650,101.98,0.07,2.7218,3.37,200101
1,600600,2001-02-28,1.358274e+08,2.4402,-0.0950,0.001650,104.46,0.07,2.7218,2.85,200102
2,600600,2001-03-30,1.377977e+09,22.1769,0.0865,0.001650,113.50,0.07,2.7218,2.85,200103
3,600600,2001-04-30,5.551689e+08,8.3685,-0.0383,0.001650,102.51,0.11,4.2589,5.77,200104
4,600600,2001-05-31,7.128193e+08,10.4741,0.0992,0.001650,112.68,0.11,4.2589,5.77,200105
...,...,...,...,...,...,...,...,...,...,...,...
259,600600,2022-08-31,1.355267e+10,18.1529,0.0963,0.001366,41.02,2.09,11.6417,27.17,202208
260,600600,2022-09-30,1.111800e+10,15.4136,-0.0167,0.001342,40.34,2.09,11.6417,27.17,202209
261,600600,2022-10-31,8.778375e+09,13.0952,-0.2298,0.001413,33.07,3.13,16.3888,41.04,202210
262,600600,2022-11-30,1.278207e+10,18.7276,0.2384,0.001676,40.95,3.13,16.3888,41.04,202211


In [6]:
# 读取月度beta数据
beta = pd.read_excel('./data2/beta.xls',usecols=[0,1,2])
beta.columns = ['stock','date','beta']

beta['date'] = pd.to_datetime(beta['date'])
beta['yearmonth'] = beta['date'].dt.strftime('%Y%m').astype(int)

beta

,stock,date,beta,yearmonth
0,600600,2001-01-19,1.5819,200101
1,600600,2001-02-28,1.8493,200102
2,600600,2001-03-30,1.8387,200103
3,600600,2001-04-30,1.8983,200104
4,600600,2001-05-31,1.9986,200105
...,...,...,...,...
257,600600,2022-08-31,0.9256,202208
258,600600,2022-09-30,1.0217,202209
259,600600,2022-10-31,1.3045,202210
260,600600,2022-11-30,1.6096,202211


In [7]:
# 将beta系数拼接到月度数据中
data_monthly = pd.merge(left=data_monthly, 
                       right=beta[['yearmonth', 'beta']],
                       on=['yearmonth'],
                       how='inner')
data_monthly

,stock,date,Trdsum,MonTurnR,Monret,Monrfret,PE,EPS,ROE,IncomePS,yearmonth,beta
0,600600,2001-01-19,1.919654e+08,3.1578,0.0112,0.001650,101.98,0.07,2.7218,3.37,200101,1.5819
1,600600,2001-02-28,1.358274e+08,2.4402,-0.0950,0.001650,104.46,0.07,2.7218,2.85,200102,1.8493
2,600600,2001-03-30,1.377977e+09,22.1769,0.0865,0.001650,113.50,0.07,2.7218,2.85,200103,1.8387
3,600600,2001-04-30,5.551689e+08,8.3685,-0.0383,0.001650,102.51,0.11,4.2589,5.77,200104,1.8983
4,600600,2001-05-31,7.128193e+08,10.4741,0.0992,0.001650,112.68,0.11,4.2589,5.77,200105,1.9986
...,...,...,...,...,...,...,...,...,...,...,...,...
257,600600,2022-08-31,1.355267e+10,18.1529,0.0963,0.001366,41.02,2.09,11.6417,27.17,202208,0.9256
258,600600,2022-09-30,1.111800e+10,15.4136,-0.0167,0.001342,40.34,2.09,11.6417,27.17,202209,1.0217
259,600600,2022-10-31,8.778375e+09,13.0952,-0.2298,0.001413,33.07,3.13,16.3888,41.04,202210,1.3045
260,600600,2022-11-30,1.278207e+10,18.7276,0.2384,0.001676,40.95,3.13,16.3888,41.04,202211,1.6096


In [8]:
# 计算月波动率 （月内日收益率的平方和）

# 计算每日收益率的平方
data_daily['Dret_squared'] = data_daily['Dret'] ** 2

# 按股票代码和日期进行分组，并对日收益率的平方求和
monthly_volatility = data_daily.groupby(['yearmonth'])['Dret_squared'].sum().reset_index()
monthly_volatility.columns = ['yearmonth','monthly_volatility']
monthly_volatility

,yearmonth,monthly_volatility
0,200101,0.003976
1,200102,0.013432
2,200103,0.008954
3,200104,0.003192
4,200105,0.003867
...,...,...
257,202208,0.023020
258,202209,0.006993
259,202210,0.015869
260,202211,0.025083


In [9]:
# 将月波动率拼接到月度数据中
data_monthly = pd.merge(left=data_monthly, 
                       right=monthly_volatility[['yearmonth', 'monthly_volatility']],
                       on=['yearmonth'],
                       how='inner')
data_monthly

,stock,date,Trdsum,MonTurnR,Monret,Monrfret,PE,EPS,ROE,IncomePS,yearmonth,beta,monthly_volatility
0,600600,2001-01-19,1.919654e+08,3.1578,0.0112,0.001650,101.98,0.07,2.7218,3.37,200101,1.5819,0.003976
1,600600,2001-02-28,1.358274e+08,2.4402,-0.0950,0.001650,104.46,0.07,2.7218,2.85,200102,1.8493,0.013432
2,600600,2001-03-30,1.377977e+09,22.1769,0.0865,0.001650,113.50,0.07,2.7218,2.85,200103,1.8387,0.008954
3,600600,2001-04-30,5.551689e+08,8.3685,-0.0383,0.001650,102.51,0.11,4.2589,5.77,200104,1.8983,0.003192
4,600600,2001-05-31,7.128193e+08,10.4741,0.0992,0.001650,112.68,0.11,4.2589,5.77,200105,1.9986,0.003867
...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,600600,2022-08-31,1.355267e+10,18.1529,0.0963,0.001366,41.02,2.09,11.6417,27.17,202208,0.9256,0.023020
258,600600,2022-09-30,1.111800e+10,15.4136,-0.0167,0.001342,40.34,2.09,11.6417,27.17,202209,1.0217,0.006993
259,600600,2022-10-31,8.778375e+09,13.0952,-0.2298,0.001413,33.07,3.13,16.3888,41.04,202210,1.3045,0.015869
260,600600,2022-11-30,1.278207e+10,18.7276,0.2384,0.001676,40.95,3.13,16.3888,41.04,202211,1.6096,0.025083


In [10]:
# 计算月流动性（|月收益率 / lg(月成交额)|）

# 计算月成交额的对数log
data_monthly['log_Trdsum'] = np.log(data_monthly['Trdsum'])

# 计算月流动性
data_monthly['Monthly_Liquidity'] = data_monthly.apply(lambda row: row['Monret'] / row['log_Trdsum'], axis=1)

# 取绝对值
data_monthly['Monthly_Liquidity'] = np.abs(data_monthly['Monthly_Liquidity'])

data_monthly

,stock,date,Trdsum,MonTurnR,Monret,Monrfret,PE,EPS,ROE,IncomePS,yearmonth,beta,monthly_volatility,log_Trdsum,Monthly_Liquidity
0,600600,2001-01-19,1.919654e+08,3.1578,0.0112,0.001650,101.98,0.07,2.7218,3.37,200101,1.5819,0.003976,19.072826,0.000587
1,600600,2001-02-28,1.358274e+08,2.4402,-0.0950,0.001650,104.46,0.07,2.7218,2.85,200102,1.8493,0.013432,18.726895,0.005073
2,600600,2001-03-30,1.377977e+09,22.1769,0.0865,0.001650,113.50,0.07,2.7218,2.85,200103,1.8387,0.008954,21.043883,0.004110
3,600600,2001-04-30,5.551689e+08,8.3685,-0.0383,0.001650,102.51,0.11,4.2589,5.77,200104,1.8983,0.003192,20.134783,0.001902
4,600600,2001-05-31,7.128193e+08,10.4741,0.0992,0.001650,112.68,0.11,4.2589,5.77,200105,1.9986,0.003867,20.384738,0.004866
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,600600,2022-08-31,1.355267e+10,18.1529,0.0963,0.001366,41.02,2.09,11.6417,27.17,202208,0.9256,0.023020,23.329850,0.004128
258,600600,2022-09-30,1.111800e+10,15.4136,-0.0167,0.001342,40.34,2.09,11.6417,27.17,202209,1.0217,0.006993,23.131831,0.000722
259,600600,2022-10-31,8.778375e+09,13.0952,-0.2298,0.001413,33.07,3.13,16.3888,41.04,202210,1.3045,0.015869,22.895557,0.010037
260,600600,2022-11-30,1.278207e+10,18.7276,0.2384,0.001676,40.95,3.13,16.3888,41.04,202211,1.6096,0.025083,23.271309,0.010244


In [11]:
# 计算月股价高点（当月股价最高值与前三个月股价的最大值的比值，用日度数据计算）
data_daily_copy = data_daily.copy() 

data_daily_copy['monthly_high'] = data_daily_copy.groupby('yearmonth')['close'].transform('max')

data_daily_copy = data_daily_copy[['yearmonth','monthly_high']]
# 去除重复的数据，并在原地修改 data_daily
data_daily_copy.drop_duplicates(subset=['yearmonth', 'monthly_high'], inplace=True)

# 计算前三个月的最大股价
data_daily_copy['three_month_max'] = data_daily_copy['monthly_high'].rolling(window=3,min_periods=1).max()


data_daily_copy

,yearmonth,monthly_high,three_month_max
0,200505,9.20,9.20
2,200506,9.35,9.35
24,200507,8.92,9.35
45,200508,9.45,9.45
68,200509,9.20,9.45
...,...,...,...
5259,202208,109.50,109.50
5282,202209,106.95,109.50
5303,202210,104.06,109.50
5319,202211,101.34,106.95


In [12]:
data_daily_copy['monthly_highPrice'] = data_daily_copy.apply(lambda row: row['monthly_high'] / row['three_month_max'], axis=1)

data_daily_copy

,yearmonth,monthly_high,three_month_max,monthly_highPrice
0,200505,9.20,9.20,1.000000
2,200506,9.35,9.35,1.000000
24,200507,8.92,9.35,0.954011
45,200508,9.45,9.45,1.000000
68,200509,9.20,9.45,0.973545
...,...,...,...,...
5259,202208,109.50,109.50,1.000000
5282,202209,106.95,109.50,0.976712
5303,202210,104.06,109.50,0.950320
5319,202211,101.34,106.95,0.947546


In [13]:
# 将月股价高点拼接到月度数据上
data_monthly = pd.merge(left=data_monthly, 
                       right=data_daily_copy[['yearmonth', 'monthly_highPrice']],
                       on=['yearmonth'],
                       how='inner')
data_monthly

,stock,date,Trdsum,MonTurnR,Monret,Monrfret,PE,EPS,ROE,IncomePS,yearmonth,beta,monthly_volatility,log_Trdsum,Monthly_Liquidity,monthly_highPrice
0,600600,2001-01-19,1.919654e+08,3.1578,0.0112,0.001650,101.98,0.07,2.7218,3.37,200101,1.5819,0.003976,19.072826,0.000587,1.000000
1,600600,2001-02-28,1.358274e+08,2.4402,-0.0950,0.001650,104.46,0.07,2.7218,2.85,200102,1.8493,0.013432,18.726895,0.005073,0.889085
2,600600,2001-03-30,1.377977e+09,22.1769,0.0865,0.001650,113.50,0.07,2.7218,2.85,200103,1.8387,0.008954,21.043883,0.004110,0.907570
3,600600,2001-04-30,5.551689e+08,8.3685,-0.0383,0.001650,102.51,0.11,4.2589,5.77,200104,1.8983,0.003192,20.134783,0.001902,1.000000
4,600600,2001-05-31,7.128193e+08,10.4741,0.0992,0.001650,112.68,0.11,4.2589,5.77,200105,1.9986,0.003867,20.384738,0.004866,1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,600600,2022-08-31,1.355267e+10,18.1529,0.0963,0.001366,41.02,2.09,11.6417,27.17,202208,0.9256,0.023020,23.329850,0.004128,1.000000
258,600600,2022-09-30,1.111800e+10,15.4136,-0.0167,0.001342,40.34,2.09,11.6417,27.17,202209,1.0217,0.006993,23.131831,0.000722,0.976712
259,600600,2022-10-31,8.778375e+09,13.0952,-0.2298,0.001413,33.07,3.13,16.3888,41.04,202210,1.3045,0.015869,22.895557,0.010037,0.950320
260,600600,2022-11-30,1.278207e+10,18.7276,0.2384,0.001676,40.95,3.13,16.3888,41.04,202211,1.6096,0.025083,23.271309,0.010244,0.947546


In [14]:
# 计算月已实现偏度（使用文献中改写的日已实现偏度公式计算，即将公式中的分钟数据换为日度数据）
data_daily_cal = data_daily[['yearmonth','Dret']].copy()
data_daily_cal['Dret_squared'] = data_daily_cal['Dret'] ** 2
data_daily_cal['Dret_tri'] = data_daily_cal['Dret'] ** 3
RDVar = data_daily_cal.groupby(['yearmonth'])['Dret_squared'].sum().reset_index()
RDVar.columns = ['yearmonth','Dret_squared_sum']
RDVar

,yearmonth,Dret_squared_sum
0,200101,0.003976
1,200102,0.013432
2,200103,0.008954
3,200104,0.003192
4,200105,0.003867
...,...,...
257,202208,0.023020
258,202209,0.006993
259,202210,0.015869
260,202211,0.025083


In [15]:
Trible = data_daily_cal.groupby(['yearmonth'])['Dret_tri'].sum().reset_index()
Trible.columns = ['yearmonth','Dret_tri_sum']
Trible

,yearmonth,Dret_tri_sum
0,200101,-0.000031
1,200102,-0.000388
2,200103,0.000357
3,200104,0.000046
4,200105,0.000070
...,...,...
257,202208,0.001354
258,202209,-0.000001
259,202210,-0.000867
260,202211,0.001230


In [16]:
monthly_days = data_daily_cal.groupby('yearmonth').size()
monthly_days_df = monthly_days.reset_index(name='days')
monthly_days_df

,yearmonth,days
0,200101,14
1,200102,13
2,200103,22
3,200104,21
4,200105,18
...,...,...
257,202208,23
258,202209,21
259,202210,16
260,202211,22


In [17]:
# 将上面计算的部分数据组合
monthly_days_df = pd.merge(left=monthly_days_df, 
                       right=Trible[['yearmonth', 'Dret_tri_sum']],
                       on=['yearmonth'],
                       how='inner')

monthly_days_df = pd.merge(left=monthly_days_df, 
                       right=RDVar[['yearmonth', 'Dret_squared_sum']],
                       on=['yearmonth'],
                       how='inner')

monthly_days_df

,yearmonth,days,Dret_tri_sum,Dret_squared_sum
0,200101,14,-0.000031,0.003976
1,200102,13,-0.000388,0.013432
2,200103,22,0.000357,0.008954
3,200104,21,0.000046,0.003192
4,200105,18,0.000070,0.003867
...,...,...,...,...
257,202208,23,0.001354,0.023020
258,202209,21,-0.000001,0.006993
259,202210,16,-0.000867,0.015869
260,202211,22,0.001230,0.025083


In [18]:
import math
monthly_days_df['RSkew'] = monthly_days_df.apply(lambda row: math.sqrt(row['days']) * row['Dret_tri_sum'] / (row['Dret_squared_sum']**1.5) , axis=1)

monthly_days_df

,yearmonth,days,Dret_tri_sum,Dret_squared_sum,RSkew
0,200101,14,-0.000031,0.003976,-0.463335
1,200102,13,-0.000388,0.013432,-0.898377
2,200103,22,0.000357,0.008954,1.974930
3,200104,21,0.000046,0.003192,1.162366
4,200105,18,0.000070,0.003867,1.241055
...,...,...,...,...,...
257,202208,23,0.001354,0.023020,1.859462
258,202209,21,-0.000001,0.006993,-0.010022
259,202210,16,-0.000867,0.015869,-1.735036
260,202211,22,0.001230,0.025083,1.452349


In [19]:
# 将月已实现偏度加入月度数据当中
data_monthly = pd.merge(left=data_monthly, 
                       right=monthly_days_df[['yearmonth', 'RSkew']],
                       on=['yearmonth'],
                       how='inner')
data_monthly

,stock,date,Trdsum,MonTurnR,Monret,Monrfret,PE,EPS,ROE,IncomePS,yearmonth,beta,monthly_volatility,log_Trdsum,Monthly_Liquidity,monthly_highPrice,RSkew
0,600600,2001-01-19,1.919654e+08,3.1578,0.0112,0.001650,101.98,0.07,2.7218,3.37,200101,1.5819,0.003976,19.072826,0.000587,1.000000,-0.463335
1,600600,2001-02-28,1.358274e+08,2.4402,-0.0950,0.001650,104.46,0.07,2.7218,2.85,200102,1.8493,0.013432,18.726895,0.005073,0.889085,-0.898377
2,600600,2001-03-30,1.377977e+09,22.1769,0.0865,0.001650,113.50,0.07,2.7218,2.85,200103,1.8387,0.008954,21.043883,0.004110,0.907570,1.974930
3,600600,2001-04-30,5.551689e+08,8.3685,-0.0383,0.001650,102.51,0.11,4.2589,5.77,200104,1.8983,0.003192,20.134783,0.001902,1.000000,1.162366
4,600600,2001-05-31,7.128193e+08,10.4741,0.0992,0.001650,112.68,0.11,4.2589,5.77,200105,1.9986,0.003867,20.384738,0.004866,1.000000,1.241055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,600600,2022-08-31,1.355267e+10,18.1529,0.0963,0.001366,41.02,2.09,11.6417,27.17,202208,0.9256,0.023020,23.329850,0.004128,1.000000,1.859462
258,600600,2022-09-30,1.111800e+10,15.4136,-0.0167,0.001342,40.34,2.09,11.6417,27.17,202209,1.0217,0.006993,23.131831,0.000722,0.976712,-0.010022
259,600600,2022-10-31,8.778375e+09,13.0952,-0.2298,0.001413,33.07,3.13,16.3888,41.04,202210,1.3045,0.015869,22.895557,0.010037,0.950320,-1.735036
260,600600,2022-11-30,1.278207e+10,18.7276,0.2384,0.001676,40.95,3.13,16.3888,41.04,202211,1.6096,0.025083,23.271309,0.010244,0.947546,1.452349


第二题中需要检验的factor有月市盈率（PE），月每股收益(EPS)，净资产收益(ROE),每股营业收入（IncomePS）， 月换手率（Turn）、月beta系数，月波动率，月流动性，月股价高点，月已实现偏度 共计10个factor

In [20]:
data_monthly['ExRet'] = data_monthly['Monret'] - data_monthly['Monrfret'] #ExRet（超额收益）计算超额收益率

data_monthly['PE'] = data_monthly['PE'].apply(lambda x: np.log(x))
data_monthly['MonTurnR'] = data_monthly['MonTurnR'].apply(lambda x: np.log(x))
data_monthly['IncomePS'] = data_monthly['IncomePS'].apply(lambda x: np.log(x))
data_monthly

,stock,date,Trdsum,MonTurnR,Monret,Monrfret,PE,EPS,ROE,IncomePS,yearmonth,beta,monthly_volatility,log_Trdsum,Monthly_Liquidity,monthly_highPrice,RSkew,ExRet
0,600600,2001-01-19,1.919654e+08,1.149876,0.0112,0.001650,4.624777,0.07,2.7218,1.214913,200101,1.5819,0.003976,19.072826,0.000587,1.000000,-0.463335,0.009550
1,600600,2001-02-28,1.358274e+08,0.892080,-0.0950,0.001650,4.648804,0.07,2.7218,1.047319,200102,1.8493,0.013432,18.726895,0.005073,0.889085,-0.898377,-0.096650
2,600600,2001-03-30,1.377977e+09,3.099051,0.0865,0.001650,4.731803,0.07,2.7218,1.047319,200103,1.8387,0.008954,21.043883,0.004110,0.907570,1.974930,0.084850
3,600600,2001-04-30,5.551689e+08,2.124475,-0.0383,0.001650,4.629960,0.11,4.2589,1.752672,200104,1.8983,0.003192,20.134783,0.001902,1.000000,1.162366,-0.039950
4,600600,2001-05-31,7.128193e+08,2.348906,0.0992,0.001650,4.724552,0.11,4.2589,1.752672,200105,1.9986,0.003867,20.384738,0.004866,1.000000,1.241055,0.097550
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,600600,2022-08-31,1.355267e+10,2.898830,0.0963,0.001366,3.714060,2.09,11.6417,3.302113,202208,0.9256,0.023020,23.329850,0.004128,1.000000,1.859462,0.094934
258,600600,2022-09-30,1.111800e+10,2.735250,-0.0167,0.001342,3.697344,2.09,11.6417,3.302113,202209,1.0217,0.006993,23.131831,0.000722,0.976712,-0.010022,-0.018042
259,600600,2022-10-31,8.778375e+09,2.572246,-0.2298,0.001413,3.498627,3.13,16.3888,3.714547,202210,1.3045,0.015869,22.895557,0.010037,0.950320,-1.735036,-0.231213
260,600600,2022-11-30,1.278207e+10,2.929998,0.2384,0.001676,3.712352,3.13,16.3888,3.714547,202211,1.6096,0.025083,23.271309,0.010244,0.947546,1.452349,0.236724


In [21]:
data_final = pd.concat([data_monthly[['yearmonth', 'Monret', 'Monrfret', 'ExRet',
                        'PE', 'EPS', 'ROE', 'IncomePS', 'MonTurnR', 'beta', 'monthly_volatility', 'Monthly_Liquidity', 'monthly_highPrice',
                        'RSkew']],
                        data_monthly[['PE', 'EPS', 'ROE', 'IncomePS', 'MonTurnR', 'beta', 'monthly_volatility', 'Monthly_Liquidity', 'monthly_highPrice',
                        'RSkew']].shift(periods=1)], axis=1)

data_final

,yearmonth,Monret,Monrfret,ExRet,PE,EPS,ROE,IncomePS,MonTurnR,beta,...,PE,EPS,ROE,IncomePS,MonTurnR,beta,monthly_volatility,Monthly_Liquidity,monthly_highPrice,RSkew
0,200101,0.0112,0.001650,0.009550,4.624777,0.07,2.7218,1.214913,1.149876,1.5819,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,200102,-0.0950,0.001650,-0.096650,4.648804,0.07,2.7218,1.047319,0.892080,1.8493,...,4.624777,0.07,2.7218,1.214913,1.149876,1.5819,0.003976,0.000587,1.000000,-0.463335
2,200103,0.0865,0.001650,0.084850,4.731803,0.07,2.7218,1.047319,3.099051,1.8387,...,4.648804,0.07,2.7218,1.047319,0.892080,1.8493,0.013432,0.005073,0.889085,-0.898377
3,200104,-0.0383,0.001650,-0.039950,4.629960,0.11,4.2589,1.752672,2.124475,1.8983,...,4.731803,0.07,2.7218,1.047319,3.099051,1.8387,0.008954,0.004110,0.907570,1.974930
4,200105,0.0992,0.001650,0.097550,4.724552,0.11,4.2589,1.752672,2.348906,1.9986,...,4.629960,0.11,4.2589,1.752672,2.124475,1.8983,0.003192,0.001902,1.000000,1.162366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,202208,0.0963,0.001366,0.094934,3.714060,2.09,11.6417,3.302113,2.898830,0.9256,...,3.622205,0.83,4.6618,2.563410,2.745372,0.9703,0.005562,0.001796,1.000000,0.093497
258,202209,-0.0167,0.001342,-0.018042,3.697344,2.09,11.6417,3.302113,2.735250,1.0217,...,3.714060,2.09,11.6417,3.302113,2.898830,0.9256,0.023020,0.004128,1.000000,1.859462
259,202210,-0.2298,0.001413,-0.231213,3.498627,3.13,16.3888,3.714547,2.572246,1.3045,...,3.697344,2.09,11.6417,3.302113,2.735250,1.0217,0.006993,0.000722,0.976712,-0.010022
260,202211,0.2384,0.001676,0.236724,3.712352,3.13,16.3888,3.714547,2.929998,1.6096,...,3.498627,3.13,16.3888,3.714547,2.572246,1.3045,0.015869,0.010037,0.950320,-1.735036


In [22]:
data_final.columns = ['yyyymm', 'Ret', 'Rfree', 'ExRet',
                'PE', 'EPS', 'ROE', 'IncomePS', 'MonTurnR', 'beta', 'monthly_volatility', 'Monthly_Liquidity', 'monthly_highPrice',
                'RSkew',
                'PEL1', 'EPSL1', 'ROEL1', 'IncomePSL1', 'MonTurnRL1', 'betaL1', 'monthly_volatilityL1', 'Monthly_LiquidityL1','monthly_highPriceL1',
                'RSkewL1']

data_final

,yyyymm,Ret,Rfree,ExRet,PE,EPS,ROE,IncomePS,MonTurnR,beta,...,PEL1,EPSL1,ROEL1,IncomePSL1,MonTurnRL1,betaL1,monthly_volatilityL1,Monthly_LiquidityL1,monthly_highPriceL1,RSkewL1
0,200101,0.0112,0.001650,0.009550,4.624777,0.07,2.7218,1.214913,1.149876,1.5819,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,200102,-0.0950,0.001650,-0.096650,4.648804,0.07,2.7218,1.047319,0.892080,1.8493,...,4.624777,0.07,2.7218,1.214913,1.149876,1.5819,0.003976,0.000587,1.000000,-0.463335
2,200103,0.0865,0.001650,0.084850,4.731803,0.07,2.7218,1.047319,3.099051,1.8387,...,4.648804,0.07,2.7218,1.047319,0.892080,1.8493,0.013432,0.005073,0.889085,-0.898377
3,200104,-0.0383,0.001650,-0.039950,4.629960,0.11,4.2589,1.752672,2.124475,1.8983,...,4.731803,0.07,2.7218,1.047319,3.099051,1.8387,0.008954,0.004110,0.907570,1.974930
4,200105,0.0992,0.001650,0.097550,4.724552,0.11,4.2589,1.752672,2.348906,1.9986,...,4.629960,0.11,4.2589,1.752672,2.124475,1.8983,0.003192,0.001902,1.000000,1.162366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,202208,0.0963,0.001366,0.094934,3.714060,2.09,11.6417,3.302113,2.898830,0.9256,...,3.622205,0.83,4.6618,2.563410,2.745372,0.9703,0.005562,0.001796,1.000000,0.093497
258,202209,-0.0167,0.001342,-0.018042,3.697344,2.09,11.6417,3.302113,2.735250,1.0217,...,3.714060,2.09,11.6417,3.302113,2.898830,0.9256,0.023020,0.004128,1.000000,1.859462
259,202210,-0.2298,0.001413,-0.231213,3.498627,3.13,16.3888,3.714547,2.572246,1.3045,...,3.697344,2.09,11.6417,3.302113,2.735250,1.0217,0.006993,0.000722,0.976712,-0.010022
260,202211,0.2384,0.001676,0.236724,3.712352,3.13,16.3888,3.714547,2.929998,1.6096,...,3.498627,3.13,16.3888,3.714547,2.572246,1.3045,0.015869,0.010037,0.950320,-1.735036


In [23]:
data = data_final.copy()
data

,yyyymm,Ret,Rfree,ExRet,PE,EPS,ROE,IncomePS,MonTurnR,beta,...,PEL1,EPSL1,ROEL1,IncomePSL1,MonTurnRL1,betaL1,monthly_volatilityL1,Monthly_LiquidityL1,monthly_highPriceL1,RSkewL1
0,200101,0.0112,0.001650,0.009550,4.624777,0.07,2.7218,1.214913,1.149876,1.5819,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,200102,-0.0950,0.001650,-0.096650,4.648804,0.07,2.7218,1.047319,0.892080,1.8493,...,4.624777,0.07,2.7218,1.214913,1.149876,1.5819,0.003976,0.000587,1.000000,-0.463335
2,200103,0.0865,0.001650,0.084850,4.731803,0.07,2.7218,1.047319,3.099051,1.8387,...,4.648804,0.07,2.7218,1.047319,0.892080,1.8493,0.013432,0.005073,0.889085,-0.898377
3,200104,-0.0383,0.001650,-0.039950,4.629960,0.11,4.2589,1.752672,2.124475,1.8983,...,4.731803,0.07,2.7218,1.047319,3.099051,1.8387,0.008954,0.004110,0.907570,1.974930
4,200105,0.0992,0.001650,0.097550,4.724552,0.11,4.2589,1.752672,2.348906,1.9986,...,4.629960,0.11,4.2589,1.752672,2.124475,1.8983,0.003192,0.001902,1.000000,1.162366
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
257,202208,0.0963,0.001366,0.094934,3.714060,2.09,11.6417,3.302113,2.898830,0.9256,...,3.622205,0.83,4.6618,2.563410,2.745372,0.9703,0.005562,0.001796,1.000000,0.093497
258,202209,-0.0167,0.001342,-0.018042,3.697344,2.09,11.6417,3.302113,2.735250,1.0217,...,3.714060,2.09,11.6417,3.302113,2.898830,0.9256,0.023020,0.004128,1.000000,1.859462
259,202210,-0.2298,0.001413,-0.231213,3.498627,3.13,16.3888,3.714547,2.572246,1.3045,...,3.697344,2.09,11.6417,3.302113,2.735250,1.0217,0.006993,0.000722,0.976712,-0.010022
260,202211,0.2384,0.001676,0.236724,3.712352,3.13,16.3888,3.714547,2.929998,1.6096,...,3.498627,3.13,16.3888,3.714547,2.572246,1.3045,0.015869,0.010037,0.950320,-1.735036


In [26]:
# 样本内检验
# 单因子模型：OLS线性拟合
# 需要检验的因子列表 10个
factors = ['PE','EPS','ROE','IncomePS','MonTurnR','beta','monthly_volatility','Monthly_Liquidity','monthly_highPrice','RSkew']

for factor in factors:
    factorL1 = factor + 'L1'
    formula = 'ExRet ~ ' + factorL1
    model = smf.ols(formula=formula, data=data[['ExRet',factorL1]])  # 构建模型
    results = model.fit()
    rg_con = results.params['Intercept']
    rg_con_pvalue = results.pvalues['Intercept']
    rg_factor = results.params[factorL1]
    rg_factor_pvalue = results.pvalues[factorL1]
    
    if rg_factor_pvalue <= 0.01:
        jud = '在1%的显著性水平下有样本内预测能力'
    elif (rg_factor_pvalue > 0.01) & (rg_factor_pvalue <= 0.05):
        jud = '在5%的显著性水平下有样本内预测能力'
    elif (rg_factor_pvalue > 0.05) & (rg_factor_pvalue <= 0.1):
        jud = '在10%的显著性水平下有样本内预测能力'
    else:
        jud = '无样本内预测能力'
    print('In-sample tests for one factor model with OLS:')
    print('Predictor: {:s}'.format(factor))
    print('Regressing Results: b = {:f}, k = {:f}'.format(rg_con, rg_factor))
    print('Regressing Pvalues: p = {:f}, p = {:f}'.format(rg_con_pvalue, rg_factor_pvalue))
    print('Inference: {:s}'.format(jud))
    print("-----------------------------------------------------------------------")

In-sample tests for one factor model with OLS:
Predictor: PE
Regressing Results: b = 0.088797, k = -0.020827
Regressing Pvalues: p = 0.144311, p = 0.204179
Inference: 无样本内预测能力
-----------------------------------------------------------------------
In-sample tests for one factor model with OLS:
Predictor: EPS
Regressing Results: b = 0.005733, k = 0.011269
Regressing Pvalues: p = 0.504051, p = 0.309289
Inference: 无样本内预测能力
-----------------------------------------------------------------------
In-sample tests for one factor model with OLS:
Predictor: ROE
Regressing Results: b = 0.002788, k = 0.001596
Regressing Pvalues: p = 0.814896, p = 0.371964
Inference: 无样本内预测能力
-----------------------------------------------------------------------
In-sample tests for one factor model with OLS:
Predictor: IncomePS
Regressing Results: b = -0.003353, k = 0.006651
Regressing Pvalues: p = 0.870494, p = 0.435976
Inference: 无样本内预测能力
-----------------------------------------------------------------------
In

In [27]:
def myfun_stat_gains(rout, rmean, rreal):
    #rout是模型的预测收益率，rmean是平均收益率（通常作为基准预测），rreal是实际的收益率
    R2os = 1 - np.sum((rreal-rout)**2)/np.sum((rreal-rmean)**2)
    #如果R2os大于0，这意味着预测模型比简单的平均模型能更好地解释收益率的变动
    d = (rreal - rmean)**2 - ((rreal-rout)**2 - (rmean-rout)**2)
    x = sm.add_constant(np.arange(len(d))+1)
    model = sm.OLS(d, x)
    fitres = model.fit()
    MFSEadj = fitres.tvalues[0]   #mfse的值
    pvalue_MFSEadj = fitres.pvalues[0]   #msfe的p值
    # MFSEadj是回归系数的t统计量，用于检验预测模型是否显著优于基准模型。pvalue_MFSEadj是对应的p值，用于判断统计显著性。

    if (R2os > 0) & (pvalue_MFSEadj <= 0.01):
        jud = '在1%的显著性水平下有样本外预测能力'
    elif (R2os > 0) & (pvalue_MFSEadj > 0.01) & (pvalue_MFSEadj <= 0.05):
        jud = '在5%的显著性水平下有样本外预测能力'
    elif (R2os > 0) & (pvalue_MFSEadj > 0.05) & (pvalue_MFSEadj <= 0.1):
        jud = '在10%的显著性水平下有样本外预测能力'
    else:
        jud = '无样本外预测能力'
    print('Stat gains: R2os = {:f}, MFSEadj = {:f}, MFSEpvalue = {:f}'.format(R2os, MFSEadj, pvalue_MFSEadj))
    print('Inference: {:s}'.format(jud))

    return R2os, MFSEadj, pvalue_MFSEadj

In [28]:
def myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm = 5):
    omg_out = rout/volt2/gmm #在外样本期间的超额收益率
    rp_out = rfree + omg_out*rreal # 在外样本期间的组合收益率
    Uout = np.mean(rp_out) - 0.5*gmm*np.var(rp_out) #Uout是根据投资者的风险厌恶系数gmm计算的预期效用
    omg_mean = rmean/volt2/gmm #在平均情况下的超额收益率
    rp_mean = rfree + omg_mean*rreal #在平均情况下的组合收益率
    Umean = np.mean(rp_mean) - 0.5*gmm*np.var(rp_mean) #在平均情况下的预期效用
    DeltaU = Uout - Umean #计算DeltaU，即预测模型与平均模型之间的效用差异
    #如果DeltaU接近于0（小于一个很小的阈值），则认为预测模型没有经济意义。否则，认为模型具有经济意义。
    if DeltaU < 10**-6:
        jud = '没有经济意义'
    else:
        jud = '有经济意义'
    print('Econ Gains: Delta U = {:f}, Upred = {:f}, Umean = {:f}'.format(DeltaU, Uout, Umean))
    print('Inference: {:s}'.format(jud))

    return Uout, Umean, DeltaU

In [29]:
# 样本外检验
# 单因子模型: OLS线性拟合
factors = ['PE','EPS','ROE','IncomePS','MonTurnR','beta','monthly_volatility','Monthly_Liquidity','monthly_highPrice','RSkew']

def perform_out_of_sampleTest(factor,data):
    factorL1 = factor + 'L1'
    datafit = data[['yyyymm','Ret','Rfree','ExRet',factor,factorL1]].copy(deep=True)
    n_in = np.sum(datafit['yyyymm']<=201512)
    n_out = np.sum(datafit['yyyymm']>201512)
    rout = np.zeros(n_out)
    rmean = np.zeros(n_out)
    rreal = np.zeros(n_out)
    rfree = np.zeros(n_out)
    volt2 = np.zeros(n_out)
    
    for i in range(n_out):
        model = smf.ols('ExRet~'+factorL1,data=datafit[['ExRet',factorL1]].iloc[:(n_in+i),:])
        results = model.fit()
        b = results.params['Intercept']
        k = results.params[factorL1]
        f = datafit[factor].iloc[n_in+i-1]
        rreal[i] = datafit['ExRet'].iloc[n_in+i]
        rfree[i] = datafit['Rfree'].iloc[n_in+i]
        rout[i] = k*f+b
        rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
        volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)
        
    print()
    print('Out-of-sample tests for one factor model with OLS:')
    print('Predictor: {:s}'.format(factor))
    R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
    Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
    print('---------------------------------------------------------------------')
    del datafit

for factor in factors:
    perform_out_of_sampleTest(factor,data)


Out-of-sample tests for one factor model with OLS:
Predictor: PE
Stat gains: R2os = 0.004758, MFSEadj = -0.848340, MFSEpvalue = 0.398718
Inference: 无样本外预测能力
Econ Gains: Delta U = 0.000054, Upred = 0.002818, Umean = 0.002765
Inference: 有经济意义
---------------------------------------------------------------------

Out-of-sample tests for one factor model with OLS:
Predictor: EPS
Stat gains: R2os = -0.010055, MFSEadj = -0.244693, MFSEpvalue = 0.807305
Inference: 无样本外预测能力
Econ Gains: Delta U = -0.000073, Upred = 0.002692, Umean = 0.002765
Inference: 没有经济意义
---------------------------------------------------------------------

Out-of-sample tests for one factor model with OLS:
Predictor: ROE
Stat gains: R2os = -0.001986, MFSEadj = -0.126994, MFSEpvalue = 0.899256
Inference: 无样本外预测能力
Econ Gains: Delta U = 0.000007, Upred = 0.002772, Umean = 0.002765
Inference: 有经济意义
---------------------------------------------------------------------

Out-of-sample tests for one factor model with OLS:
Predic

In [30]:
# 样本外检验
# 多因子模型：OLS线性拟合

factor_out = 'PE, EPS, ROE, IncomePS, MonTurnR, beta, monthly_volatility, Monthly_Liquidity, monthly_highPrice, RSkew'
datafit = data.copy(deep=True)

n_in = np.sum(datafit['yyyymm'] <= 201512)
n_out = np.sum(datafit['yyyymm'] > 201512)
rout = np.zeros(n_out)
rmean = np.zeros(n_out)
rreal = np.zeros(n_out)
rfree = np.zeros(n_out)
volt2 = np.zeros(n_out)

for i in range(n_out):
    model = smf.ols('ExRet ~ PEL1 + EPSL1 + ROEL1 + IncomePSL1 + MonTurnRL1 + betaL1 + monthly_volatilityL1 + '
                    'Monthly_LiquidityL1 + monthly_highPriceL1 + RSkewL1',
                    data=datafit[['ExRet', 'PEL1', 'EPSL1', 'ROEL1', 'IncomePSL1', 'MonTurnRL1', 'betaL1',
                                  'monthly_volatilityL1', 'Monthly_LiquidityL1', 'monthly_highPriceL1', 'RSkewL1']].iloc[:(n_in+i), :])
    results = model.fit()
    k = results.params.values
    f = datafit[['PE', 'EPS', 'ROE', 'IncomePS', 'MonTurnR', 'beta', 'monthly_volatility', 'Monthly_Liquidity',
                 'monthly_highPrice', 'RSkew']].iloc[n_in+i-1, :].values
    f = np.concatenate((np.array([1]), f))
    rreal[i] = datafit['ExRet'].iloc[n_in+i]
    rfree[i] = datafit['Rfree'].iloc[n_in+i]
    rout[i] = np.sum(k*f)
    rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
    volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)

print()
print('Out-of-sample tests for multi-factor model with OLS:')
print('Predictor: {:s}'.format(factor_out))
R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
del datafit


Out-of-sample tests for multi-factor model with OLS:
Predictor: PE, EPS, ROE, IncomePS, MonTurnR, beta, monthly_volatility, Monthly_Liquidity, monthly_highPrice, RSkew
Stat gains: R2os = -0.057672, MFSEadj = -0.852538, MFSEpvalue = 0.396399
Inference: 无样本外预测能力
Econ Gains: Delta U = -0.000802, Upred = 0.001963, Umean = 0.002765
Inference: 没有经济意义


In [32]:
# 样本外检验
# 多因子模型：LASSO回归, Ridge回归，ElasticNet回归

from sklearn.preprocessing import StandardScaler
#Ridge回归
factor_out = 'PE, EPS, ROE, IncomePS, MonTurnR, beta, monthly_volatility, Monthly_Liquidity, monthly_highPrice, RSkew'
factor_list = np.array(['PE', 'EPS', 'ROE', 'IncomePS', 'MonTurnR', 'beta', 'monthly_volatility', 'Monthly_Liquidity', 'monthly_highPrice', 'RSkew'])

datafit = data.copy(deep=True)
datafit.dropna(inplace=True)

n_in = np.sum(datafit['yyyymm'] <= 201512)
n_out = np.sum(datafit['yyyymm'] > 201512)
rout = np.zeros(n_out)
rmean = np.zeros(n_out)
rreal = np.zeros(n_out)
rfree = np.zeros(n_out)
volt2 = np.zeros(n_out)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(datafit[['PEL1', 'EPSL1', 'ROEL1', 'IncomePSL1', 'MonTurnRL1', 'betaL1','monthly_volatilityL1', 'Monthly_LiquidityL1', 'monthly_highPriceL1', 'RSkewL1']])


reg = sklm.RidgeCV(cv=10, fit_intercept=True)

for i in range(n_out):
    X = X_scaled[:(n_in+i), :]
    y = datafit['ExRet'].iloc[:(n_in+i)].values
    reg.fit(X, y)
    # print(factor_list[np.abs(reg.coef_) != 0])
    k = np.concatenate((np.array([reg.intercept_]), reg.coef_))
    f = datafit[['PE', 'EPS', 'ROE', 'IncomePS', 'MonTurnR', 'beta', 'monthly_volatility', 'Monthly_Liquidity',
                 'monthly_highPrice', 'RSkew']].iloc[n_in+i-1, :].values
    f = np.concatenate((np.array([1]), f))
    rreal[i] = datafit['ExRet'].iloc[n_in+i]
    rfree[i] = datafit['Rfree'].iloc[n_in+i]
    rout[i] = np.sum(k*f)
    rmean[i] = np.mean(datafit['ExRet'].iloc[:(n_in+i)].values)
    volt2[i] = np.sum(datafit['Ret'].iloc[(n_in+i-12):(n_in+i)].values**2)

print()
print('Out-of-sample tests for multi-factor model with ML method:')
print('Predictor: {:s}'.format(factor_out))
R2os, MFSEadj, pvalue_MFSEadj = myfun_stat_gains(rout, rmean, rreal)
Uout, Umean, DeltaU = myfun_econ_gains(rout, rmean, rreal, rfree, volt2, gmm=5)
del datafit


Out-of-sample tests for multi-factor model with ML method:
Predictor: PE, EPS, ROE, IncomePS, MonTurnR, beta, monthly_volatility, Monthly_Liquidity, monthly_highPrice, RSkew
Stat gains: R2os = -0.064471, MFSEadj = -0.198923, MFSEpvalue = 0.842815
Inference: 无样本外预测能力
Econ Gains: Delta U = 0.001144, Upred = 0.003908, Umean = 0.002764
Inference: 有经济意义
